# Medical LLM Fine-Tuning — Qwen2.5-1.5B-Instruct + PubMedQA (QLoRA)

**Stack (pinned, tested together on a Kaggle T4 x1, 16GB):**
- `transformers` 5.13.0 · `trl` 1.7.1 · `peft` 0.19.1 · `accelerate` 1.14.0 · `bitsandbytes` 0.49.2 · `datasets` 5.0.0
- Base model: `Qwen/Qwen2.5-1.5B-Instruct`
- Data: `qiaojin/PubMedQA` — `pqa_artificial` subset for training, `pqa_labeled` for held-out evaluation (the standard PubMedQA train/eval split used in the original paper)
- 4-bit QLoRA, gradient checkpointing, paged optimizer → fits comfortably in T4 VRAM (peak ~6-7GB)

**Kaggle setup before running:**
1. Settings → Accelerator → **GPU T4 x1**
2. Settings → Internet → **On**
3. Add-ons → Secrets → add a secret named `HF_TOKEN` with a Hugging Face **write** token (only needed for the push-to-hub cell; the rest of the notebook runs fine without it)


## 1. GPU check

In [1]:
!nvidia-smi

Mon Jul  6 16:59:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install libraries (pinned, single pass, no conflicting reinstalls)

We install **exact pinned versions** so the notebook behaves the same every run. Everything is installed in one `pip` call to let the resolver settle on one consistent set instead of overwriting itself twice (this was the main source of hidden version drift in the original notebook).


In [2]:
%%capture
!pip install -q -U \
    "transformers==5.13.0" \
    "trl==1.7.1" \
    "peft==0.19.1" \
    "accelerate==1.14.0" \
    "bitsandbytes==0.49.2" \
    "datasets==5.0.0" \
    "huggingface_hub>=0.36.0" \
    "evaluate==0.4.6" \
    "rouge_score==0.1.2" \
    "sentencepiece==0.2.0"


In [3]:
# Sanity-check the installed versions before we touch the GPU
import torch, transformers, trl, peft, accelerate, bitsandbytes, datasets

print("Torch:        ", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("Transformers: ", transformers.__version__)
print("TRL:          ", trl.__version__)
print("PEFT:         ", peft.__version__)
print("Accelerate:   ", accelerate.__version__)
print("BitsAndBytes: ", bitsandbytes.__version__)
print("Datasets:     ", datasets.__version__)

if torch.cuda.is_available():
    print("GPU:          ", torch.cuda.get_device_name(0))
    print("BF16 support: ", torch.cuda.is_bf16_supported())
else:
    raise RuntimeError("No GPU detected — set Settings > Accelerator > GPU T4 x1 and rerun.")


Torch:         2.10.0+cu128 | CUDA available: True
Transformers:  5.13.0
TRL:           1.7.1
PEFT:          0.19.1
Accelerate:    1.14.0
BitsAndBytes:  0.49.2
Datasets:      5.0.0
GPU:           Tesla T4
BF16 support:  True


## 3. Imports

In [4]:
import os
import gc
import torch
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

torch.manual_seed(42)
np.random.seed(42)


## 4. (Optional) Hugging Face login

Reads `HF_TOKEN` from Kaggle Secrets if available, otherwise falls back to an interactive prompt. This cell **never crashes the notebook** if no token is provided — it just disables the push-to-hub step later.


In [5]:
HF_TOKEN = None

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("Loaded HF_TOKEN from Kaggle Secrets.")
except Exception:
    try:
        HF_TOKEN = os.environ.get("HF_TOKEN")
        if HF_TOKEN:
            print("Loaded HF_TOKEN from environment.")
    except Exception:
        pass

PUSH_TO_HUB = HF_TOKEN is not None

if PUSH_TO_HUB:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face Hub. push_to_hub is ENABLED.")
else:
    print("No HF_TOKEN found. push_to_hub is DISABLED — training/eval/inference still run normally.")


No HF_TOKEN found. push_to_hub is DISABLED — training/eval/inference still run normally.


## 5. Config

Change `HF_USERNAME` / `HUB_MODEL_ID` if you want to push the adapter to your own Hugging Face account.


In [6]:
MODEL_NAME       = "Qwen/Qwen2.5-1.5B-Instruct"
HF_USERNAME      = "jakirniloy"          # <-- change this
HUB_MODEL_ID     = f"{HF_USERNAME}/qwen2.5-1.5b-pubmedqa-qlora"

OUTPUT_DIR       = "/kaggle/working/medical_qwen"
MAX_SEQ_LEN      = 512          # PubMedQA contexts are long; 512 covers almost all examples without truncating the answer
TRAIN_SUBSET     = 3000         # subset of pqa_artificial used for training — keeps a full run well under Kaggle's session limits
NUM_EPOCHS       = 2


## 6. Load PubMedQA

- `pqa_artificial` (211k rows, PubMed-generated QA pairs) → training subset
- `pqa_labeled` (1k rows, expert-annotated) → held-out evaluation, split 90/10 into eval/test

This mirrors how the PubMedQA paper itself splits the data: artificial for training, labeled for evaluation.


In [7]:
raw_train = load_dataset("qiaojin/PubMedQA", "pqa_artificial", split="train")
raw_train = raw_train.shuffle(seed=42).select(range(TRAIN_SUBSET))

raw_labeled = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
raw_labeled = raw_labeled.shuffle(seed=42)
split_labeled = raw_labeled.train_test_split(test_size=0.2, seed=42)
raw_eval  = split_labeled["train"]   # used for eval_loss during training
raw_test  = split_labeled["test"]    # held out for the final qualitative + ROUGE evaluation

print(raw_train)
print(raw_eval)
print(raw_test)
raw_train[0]


README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

pqa_artificial/train-00000-of-00001.parq(…):   0%|          | 0.00/233M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/211269 [00:00<?, ? examples/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
    num_rows: 3000
})
Dataset({
    features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
    num_rows: 800
})
Dataset({
    features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
    num_rows: 200
})


{'pubid': 15248378,
 'question': 'Are keratin 8 Y54H and G62C mutations associated with inflammatory bowel disease?',
 'context': {'contexts': ['Keratin 8 is a major component of intermediate filaments in single-layered epithelia of the gastrointestinal tract. Keratin 8 deficient mice display signs of colitis and diarrhoea characteristic for inflammatory bowel disease. Very recently, two keratin 8 mutations, Y54H and G62C, were identified.',
   'We investigated if these keratin 8 missense mutations were associated with inflammatory bowel disease.',
   "In total, 217 German patients with Crohn' s disease, 131 German patients with ulcerative colitis, and 560 German control subjects were enrolled in this study.",
   'Samples were analysed by PCR amplification and subsequent melting curve analysis using fluorescence resonance energy transfer probes.',
   "The G62C mutation was detected in five (2.3%) patients presenting with Crohn's disease and in three (2.3%) with ulcerative colitis. In c

## 7. Format examples into a single `text` field

In [8]:
def formatting(example):
    context = "\n".join(example["context"]["contexts"])
    text = (
        "### Instruction\n"
        "You are a professional medical assistant. Answer the question using only the given context.\n\n"
        f"### Context\n{context}\n\n"
        f"### Question\n{example['question']}\n\n"
        f"### Answer\n{example['long_answer']}"
    )
    return {"text": text}

train_dataset = raw_train.map(formatting, remove_columns=raw_train.column_names)
eval_dataset  = raw_eval.map(formatting,  remove_columns=raw_eval.column_names)

print(train_dataset[0]["text"][:800])


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

### Instruction
You are a professional medical assistant. Answer the question using only the given context.

### Context
Keratin 8 is a major component of intermediate filaments in single-layered epithelia of the gastrointestinal tract. Keratin 8 deficient mice display signs of colitis and diarrhoea characteristic for inflammatory bowel disease. Very recently, two keratin 8 mutations, Y54H and G62C, were identified.
We investigated if these keratin 8 missense mutations were associated with inflammatory bowel disease.
In total, 217 German patients with Crohn' s disease, 131 German patients with ulcerative colitis, and 560 German control subjects were enrolled in this study.
Samples were analysed by PCR amplification and subsequent melting curve analysis using fluorescence resonance energy t


## 8. Tokenizer

`padding_side="right"` is required for causal-LM training (left padding is only for generation).


In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## 9. Load the model in 4-bit (QLoRA)

Transformers 5.x deprecated `torch_dtype` in favor of `dtype`. We use `device_map="auto"` so Accelerate places the quantized model on the GPU — **do not** call `.to("cuda")` on a 4-bit model, that path is what caused the OOM/crash in the original notebook (it forces an extra full-precision copy during the device move).


In [10]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    dtype=torch.bfloat16,
    device_map={"": 0},
)

model.config.use_cache = False  # required when using gradient checkpointing
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## 10. LoRA config

In [11]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)


## 11. Training config

T4-safe defaults: small per-device batch with gradient accumulation, gradient checkpointing, bf16 (matches the bf16 compute dtype used for quantization — mixing fp16 training with a bf16 compute dtype, like the original notebook did, is what silently wastes memory and can destabilize loss), and a paged 8-bit optimizer to keep the optimizer state off the GPU as much as possible.


In [12]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=MAX_SEQ_LEN,
    packing=False,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    num_train_epochs=NUM_EPOCHS,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,

    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit",

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",

    push_to_hub=PUSH_TO_HUB,
    hub_model_id=HUB_MODEL_ID if PUSH_TO_HUB else None,
)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## 12. Trainer

In [13]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

trainer.model.print_trainable_parameters()


Adding EOS to train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 13. Train

In [14]:
torch.cuda.empty_cache()
gc.collect()




519

In [15]:
train_result = trainer.train()
print(train_result.metrics)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,3.304435,3.489322,1.724656,1267153.000000,0.593690
2,3.212296,3.493495,1.700641,2534306.000000,0.593138


{'train_runtime': 17912.52, 'train_samples_per_second': 0.335, 'train_steps_per_second': 0.01, 'total_flos': 2.3626882455404544e+16, 'train_loss': 3.2749086237968283, 'epoch': 2.0}


## 14. Save the adapter locally

In [16]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved to", OUTPUT_DIR)


Saved to /kaggle/working/medical_qwen


## 15. Evaluation

Two parts:
1. **Eval loss / perplexity** on the held-out `pqa_labeled` eval split (via `trainer.evaluate()`).
2. **ROUGE-L** between generated answers and reference long answers on the untouched `raw_test` split, plus a few printed qualitative examples.


In [17]:
eval_metrics = trainer.evaluate()0.
eval_loss = eval_metrics["eval_loss"]
print(f"Eval loss:       {eval_loss:.4f}")
print(f"Eval perplexity: {np.exp(eval_loss):.2f}")


Training Loss,Validation Loss,Epoch,Entropy,Num Tokens,Mean Token Accuracy
3.212296,3.493495,2,1.700641,2534306.000000,0.593138


Eval loss:       3.4935
Eval perplexity: 32.90


In [18]:
import evaluate as hf_evaluate

rouge = hf_evaluate.load("rouge")

model.config.use_cache = True  # re-enable KV cache for fast generation
model.eval()

def build_prompt(example):
    context = "\n".join(example["context"]["contexts"])
    return (
        "### Instruction\n"
        "You are a professional medical assistant. Answer the question using only the given context.\n\n"
        f"### Context\n{context}\n\n"
        f"### Question\n{example['question']}\n\n"
        "### Answer\n"
    )

N_EVAL_SAMPLES = 20
sample_indices = range(min(N_EVAL_SAMPLES, len(raw_test)))

predictions, references = [], []
for i in sample_indices:
    example = raw_test[i]
    prompt = build_prompt(example)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    predictions.append(generated.strip())
    references.append(example["long_answer"].strip())

rouge_scores = rouge.compute(predictions=predictions, references=references)
print("ROUGE scores on", len(predictions), "held-out test examples:")
for k, v in rouge_scores.items():
    print(f"  {k}: {v:.4f}")

print("\n--- Sample predictions ---")
for i in range(min(3, len(predictions))):
    print(f"\nQ: {raw_test[i]['question']}")
    print(f"Reference: {references[i][:300]}")
    print(f"Generated: {predictions[i][:300]}")


ROUGE scores on 20 held-out test examples:
  rouge1: 0.2733
  rouge2: 0.0914
  rougeL: 0.2014
  rougeLsum: 0.2032

--- Sample predictions ---

Q: Is there a relationship between serum paraoxonase level and epicardial fat tissue thickness?
Reference: Serum PON 1 level is not correlated with the epicardial fat tissue thickness. But PON 1 level is lower in patients with epicardial fat tissue thickness 7 mm and over. Therefore, increased atherosclerosis progression can be found among patients with 7 mm and higher epicardial fat tissue thickness.
Generated: Serum PON 1 level is decreased with increasing epicardial fat tissue thickness. This may indicate that PON 1 plays an important role in the regulation of adipose tissue metabolism.

Q: Expression of c-kit protooncogen in hepatitis B virus-induced chronic hepatitis, cirrhosis and hepatocellular carcinoma: has it a diagnostic role?
Reference: C-kit positivity was observed in the mitotic, proliferating and also dysplastic hepatic cells. The

## 16. Push to Hugging Face Hub

Runs only if `HF_TOKEN` was found in step 4.


In [19]:
if PUSH_TO_HUB:
    trainer.push_to_hub(commit_message="QLoRA fine-tune of Qwen2.5-1.5B-Instruct on PubMedQA")
    tokenizer.push_to_hub(HUB_MODEL_ID)
    print(f"Pushed to https://huggingface.co/{HUB_MODEL_ID}")
else:
    print("Skipped: no HF_TOKEN found in step 4. The adapter is still saved locally at", OUTPUT_DIR)


Skipped: no HF_TOKEN found in step 4. The adapter is still saved locally at /kaggle/working/medical_qwen


## 17. Inference (loading the saved adapter fresh)

Loads the base model in 4-bit again and attaches the saved LoRA adapter with `PeftModel`, simulating how you'd use the model in a new session/notebook.


In [20]:
import gc
del model, trainer
gc.collect()
torch.cuda.empty_cache()

from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    dtype=torch.bfloat16,
    device_map="auto",
)
inference_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
inference_model.eval()

inference_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [21]:
def ask(question, context_paragraphs):
    context = "\n".join(context_paragraphs)
    prompt = (
        "### Instruction\n"
        "You are a professional medical assistant. Answer the question using only the given context.\n\n"
        f"### Context\n{context}\n\n"
        f"### Question\n{question}\n\n"
        "### Answer\n"
    )
    inputs = inference_tokenizer(prompt, return_tensors="pt").to(inference_model.device)
    with torch.no_grad():
        out = inference_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=inference_tokenizer.pad_token_id,
        )
    return inference_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


answer = ask(
    "What is diabetes?",
    ["Diabetes mellitus is a group of metabolic diseases characterized by high blood sugar levels over a prolonged period."],
)
print(answer)


The aim of this study was to investigate whether insulin resistance (IR) and hyperinsulinemia were associated with increased risk for cardiovascular disease in patients with type 2 diabetes.
A total of 1,034 patients with type 2 diabetes who had been followed up at our hospital between January 2005 and December 2007 were included in this study. The IR and hyperinsulinemia were defined as HOMA-IR > or = 2.5 and fasting plasma insulin level > or = 6 microU/mL, respectively. Multivariate logistic regression analysis was used to determine the association between IR and hyperinsulinemia and the occurrence of cardiovascular events.
During follow-up, 89 patients developed cardiovascular events including myocardial infarction, stroke, and peripheral vascular disease. After adjustment for age, sex, body mass index, smoking status, hypertension, dyslipidemia, and use of antidiabetic drugs, multivariate logistic regression analysis


In [22]:
!zip -r medical_qwen.zip /kaggle/working/medical_qwen

  adding: kaggle/working/medical_qwen/ (stored 0%)
  adding: kaggle/working/medical_qwen/tokenizer_config.json (deflated 59%)
  adding: kaggle/working/medical_qwen/training_args.bin (deflated 53%)
  adding: kaggle/working/medical_qwen/adapter_config.json (deflated 59%)
  adding: kaggle/working/medical_qwen/tokenizer.json (deflated 81%)
  adding: kaggle/working/medical_qwen/adapter_model.safetensors (deflated 21%)
  adding: kaggle/working/medical_qwen/chat_template.jinja (deflated 71%)
  adding: kaggle/working/medical_qwen/README.md (deflated 44%)
  adding: kaggle/working/medical_qwen/checkpoint-188/ (stored 0%)
  adding: kaggle/working/medical_qwen/checkpoint-188/tokenizer_config.json (deflated 59%)
  adding: kaggle/working/medical_qwen/checkpoint-188/training_args.bin (deflated 53%)
  adding: kaggle/working/medical_qwen/checkpoint-188/adapter_config.json (deflated 59%)
  adding: kaggle/working/medical_qwen/checkpoint-188/tokenizer.json (deflated 81%)
  adding: kaggle/working/medical_q

In [23]:
from IPython.display import FileLink

FileLink("/kaggle/working/medical_qwen.zip")

/kaggle/working/medical_qwen.zip

In [24]:
!ls -lh /kaggle/working

total 92M
drwxr-xr-x 3 root root 4.0K Jul  6 22:00 medical_qwen
-rw-r--r-- 1 root root  92M Jul  6 22:14 medical_qwen.zip


In [25]:
from IPython.display import FileLink
FileLink("/kaggle/working/medical_qwen.zip")

/kaggle/working/medical_qwen.zip